# **MODELO DE CLASIFICACIÓN 2:**
# **Bosque Aleatorio (** Random Forest **)**

Este es un modelo de "ensamble", que se puede entender como **el siguiente paso después de Árbol de Decisión**. En lugar de crear un solo árbol (que puede ser propenso a "memorizar" los datos, u overfitting), un **Random Forest** crea cientos de árboles (`n_estimators=100`) y les pide que "voten" por la mejor clasificación.

**Es como pedir la opinión de 100 expertos en lugar de uno solo.** Son más potentes y robustos que un solo árbol. ¡Ventaja! Al igual que los árboles de decisión, los Random Forest no requieren que escalemos los datos.


****
## **Paso 1: Introducción y Librerías**

**Objetivo:** Entrenar un modelo que pueda predecir la especie de un pingüino basándose en sus medidas físicas (largo y profundidad del pico, largo de la aleta y masa corporal). Usaremos un **Random Forest**, como el siguiente paso después .

In [ ]:
# Librerías para cargar el dataset y construir gráficos
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Librerías necesarias para el modelo DecisionTreeClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Métricas rendimiento
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

***
## **Paso 2: Cargar y Explorar los Datos**

Cargaremos el dataset penguins usando Seaborn. Es importante revisar los datos para entender su estructura y ver si hay valores faltantes (NaN).
* Cargar el dataset y Mostrar las primeras 5 filas

In [ ]:
pg = sns.load_dataset('penguins')
pg.head()

* Mostrar un resumen (tipos de datos y valores nulos)


In [ ]:
print("\n--- Información del dataset ---")
pg.info()

***
## **Paso 3: Preparación de Datos (Limpieza de datos)**

El método `.info()` mostró que hay valores nulos (NaN) en varias columnas. **Los modelos de scikit-learn no pueden trabajar con datos faltantes**. Para este ejemplo, usaremos la estrategia más simple: eliminar cualquier fila que contenga un valor nulo.

* Para este ejemplo, en lugar de imputar usaremos la estrategia más simple: eliminar cualquier fila que contenga un valor nulo.

In [ ]:
# Eliminamos filas con CUALQUIER valor NaN
pg_clean = pg.dropna()

# Comparamos el tamaño
print(f"Se eliminaron: {len(pg)-len(pg_clean)} filas con datos incompletos")

# pg_clean.info()

* Ahora convertimos variables categóricas a numéricas. `Sex` convertida a `0/1`.

In [ ]:
pg_clean['sex'] = pg_clean['sex'].map({'Male':0, 'Female':1})
# pg_clean.head()

* La columna `island` es convertida a numerica con One-Hot Encoding.

In [ ]:
pg_clean = pd.get_dummies(pg_clean, columns=['island'], prefix='island_')
# pg_clean.head()

***
## **Paso 4: Definir Features (X) y Target (y)**

Ahora, separamos nuestros datos en dos partes:

* **Features (`X`):** Las variables de entrada, las "preguntas" que le damos al modelo. Usaremos solo las medidas numéricas.

* **Target (`y`):** La variable de salida, la "respuesta" que queremos predecir (En este caso, las columna `species`).

In [ ]:
pg_clean.head()

* Lista de columnas que usaremos como features y definimos `X`

In [ ]:
features_list = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g','sex','island__Biscoe','island__Dream','island__Torgersen']
X = pg_clean[features_list]
#X.head()

* Definimos la columna (o lista de columnas) que será nuestro target `y`

In [ ]:
y = pg_clean['species']
#y.head()

In [ ]:
# Mostramos las "clases" o especies únicas que el modelo debe aprender
print(f"Features seleccionadas (X): {features_list}")
print(f"Target (y): 'species'")
print(f"Clases a predecir: {y.unique()}")

***
## **Paso 5: Dividir los datos en Train y Test**

Este es el paso más importante del Machine Learning. El set de datos se divide entre datos para entrenar el modelo y el resto para evaluarlo. Generalmente la relación entre datos de entrenamiento y evaluación es **80-20%** o **70-30%**.

* **Set de Entrenamiento (Train):** Lo usaremos para "entrenar" al modelo (80% de los datos).

*  **Set de Prueba (Test):** Lo guardaremos para "evaluar" al modelo con datos que nunca ha visto (20% de los datos).

* Usamos `random_state=42` para que la división aleatoria sea siempre la misma (reproducible).

In [ ]:
# Dividimos los datos (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Total de datos de entrenamiento (train): {len(X_train)}")
print(f"Total de datos de prueba (test): {len(X_test)}")

***
## **Paso 6: Crear y Entrenar el Modelo**

Inicializamos el modelo **Random Forest**. Crearemos un <font color="red">**bosque de 100 árboles**</font> usando `n_estimators=100`.

* Entrenamos el modelo usando el método `.fit()` solo con los datos de entrenamiento.

In [ ]:
# 1. Inicializar el modelo
modelo_rf = RandomForestClassifier(max_depth=3,n_estimators=100, random_state=42)

# 2. Entrenar el modelo
modelo_rf.fit(X_train, y_train)

## <font color="red">**CUIDADO**</font>

* El modelo ya está entrenado. La siguiente celda la usaremos SOLO para evaluar cuanto demora en entrenarse el modelo. **No es necesaria** para ejecutar el modelo

In [ ]:
%%timeit
modelo_rf.fit(X_train, y_train)

****
## **Paso 7: Evaluar el Modelo**

Ahora, usamos los datos de prueba (**X_test**) que el modelo nunca vio.

1. Hacemos predicciones con los datos de prueba usando `.predict()`

In [ ]:
y_pred = modelo_rf.predict(X_test)

2. Comparamos esas predicciones (**y_pred**) con las respuestas correctas(**y_test**). Primero calculamos la **precisión** usando `.accuracy_score()`:

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Precisión (Accuracy) del modelo: {accuracy * 100:.1f}%")

3. Comparamos esas predicciones (**y_pred**) con las respuestas correctas(**y_test**). Ahora calculamos la **matriz de confusión** usando usando `.confusion_matrix()`:

In [ ]:
print("--- Matriz de Confusión ---")
print("\n(Las filas son el valor REAL, las columnas la PREDICCIÓN) \n")

labels = modelo_rf.classes_
cm = confusion_matrix(y_test, y_pred, labels=labels)

# La mostramos como un DataFrame para que sea más fácil de leer
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
print(cm_df)


***
## **Paso 8: Importancia de las Características (** Feature Importance **)**

* Obtenemos las importancias y creamos una serie de Pandas para verlas fácilmente

In [ ]:
importances = modelo_rf.feature_importances_
feature_names = X_train.columns
forest_importances = pd.Series(importances, index=feature_names)

# Ordenamos de mayor a menor importancia
forest_importances_sorted = forest_importances.sort_values(ascending=False)

print("--- Importancia de las Características (Random Forest) ---\n")
print(forest_importances_sorted)

* Creamos un gráfico de barras para visualizar el resultado más facilmente

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x=forest_importances_sorted.values, y=forest_importances_sorted.index)
plt.title("Importancia de las Características en Random Forest")
plt.xlabel("Nivel de Importancia")
plt.ylabel("Característica")
plt.show()

          feature_names=features_list,       # Nombres de las features
          class_names=labels,                # Nombres de las clases (especies)
          filled=True,                       # Colorear los nodos
          rounded=True,                      # Bordes redondeados